# Etapa 1 — Exploração dos dados e tokenização

Notebook para rodar no **Kaggle** (Settings → Internet: On) ou localmente.

Objetivos:
1. Conhecer o HealthQA-BR (quantas questões, de quais provas, anos e áreas).
2. Gerar o corpus e o vocabulário de caracteres com `data/prepare.py`.
3. Entender como os lotes `(x, y)` são formados.
4. Calcular a perda inicial esperada, ln(tamanho do vocabulário).

In [ ]:
# No Kaggle: clone o repositório do grupo (troque a URL) e instale dependências
# !git clone https://github.com/SEU-USUARIO/mini-gpt-med.git
# %cd mini-gpt-med
!pip install -q datasets

## 1. Conhecendo o dataset

In [ ]:
from datasets import load_dataset

df = load_dataset('Larxel/healthqa-br', split='train').to_pandas()
print(df.shape)
df.head()

In [ ]:
print(df['source'].value_counts(), '\n')
print(df['year'].value_counts().sort_index(), '\n')
print(df['group'].value_counts().head(15))

In [ ]:
tam = df['question'].str.len()
print(tam.describe())
print('\nTotal de caracteres:', f"{tam.sum():,}")
print('\nExemplo:\n', df['question'].iloc[0][:800])

**Para o relatório:** anote o número de questões, a distribuição por prova e o tamanho total. A trilha pede no mínimo ~1 MB (1 milhão de caracteres).

## 2. Gerando o corpus e o vocabulário

In [ ]:
!python data/prepare.py

In [ ]:
import json
meta = json.load(open('data/processed/meta.json'))
meta

In [ ]:
from tokenizer import CharTokenizer
tok = CharTokenizer.load('data/processed/tokenizer.json')
print('Vocabulário:', tok.vocab_size)
print(repr(''.join(tok.chars)))

ids = tok.encode('Paciente com febre')
print(ids)
print(tok.decode(ids))

In [ ]:
print(open('data/processed/corpus.txt', encoding='utf-8').read()[:1500])

## 3. Formando lotes (x, y)

`y` é `x` deslocado uma posição: em cada posição o modelo tenta prever o próximo caractere.

In [ ]:
import sys, numpy as np
sys.path.insert(0, 'data')
from loader import carregar_tokens, get_batch

train = carregar_tokens('train')
x, y = get_batch(train, batch_size=4, block_size=32, rng=np.random.default_rng(0))
print(x.shape, y.shape)
for t in range(5):
    print(f'contexto {tok.decode(x[0, :t+1])!r:>12} -> alvo {tok.decode([y[0, t]])!r}')

## 4. Perda inicial esperada

Antes de treinar, o modelo prevê algo próximo de uma distribuição uniforme sobre o vocabulário. A entropia cruzada fica em −ln(1/V) = ln(V). Na semana 2, a primeira perda do modelo **tem que** bater com esse número; se não bater, há bug.

In [ ]:
import math
print(f'ln({tok.vocab_size}) = {math.log(tok.vocab_size):.4f}')